# BeeQeeper — Phase 1: Leakage-safe preprocessing

This notebook defines the **preprocessing contract** used before classical and quantum model fitting.

It intentionally does **not** perform model selection, hyperparameter optimization, threshold tuning, or final model comparison. Its purpose is to make the data transformations explicit, reproducible, and resistant to information leakage.

> **Previous phase:** [Open the full Phase 0 EDA notebook](./BeeQeeper_phase0.ipynb)

The EDA notebook should be read first when reviewing the project because the preprocessing decisions below are consequences of what was observed there.

## What Phase 0 EDA established

The preprocessing policy is based on the following findings from the completed EDA:

1. **The X10 matrix is complete.** All ten descriptors are finite and have no missing values, so there is no current need for imputation.
2. **The variables live on very different numerical scales** and several are skewed or discrete. Scale-sensitive models therefore need explicit scaling.
3. **`n_OP` is a major but not exclusive toxicity signal.** Roughly 92% of DEVELOPMENT molecules with `n_OP > 0` are toxic, compared with about 23% when `n_OP = 0`. It remains in the main X10 representation, but an `n_OP` ablation and an `n_OP = 0` subgroup analysis are pre-specified.
4. **There is meaningful feature redundancy**, especially between `TPSA_SP` and the polar-SASA descriptor (Spearman ρ ≈ 0.83). Correlation alone was not treated as sufficient reason to delete either feature.
5. **PCA should not be the default representation.** PC1 and PC2 explain the most variance but do not cleanly separate toxicity, while PC3 contains substantial label association largely through the organophosphorus dimension. Explained variance is therefore not a valid supervised feature-selection rule.
6. **Structural families matter.** Butina clusters are significantly more label-homogeneous than expected by chance, even after restricting the analysis to `n_OP = 0`. This supports preserving the frozen structure-aware folds.
7. **TEST is more novel structurally than DEVELOPMENT's ordinary internal neighborhoods**, although most TEST molecules remain inside the broad X10 physicochemical domain.
8. **X10 is a lossy structural representation.** X10 distance only moderately tracks Morgan/Tanimoto similarity, so structurally controlled evaluation remains necessary.
9. **One TEST molecule (Chitosan hydrochloride) is an extreme X10 extrapolation.** It is retained; preprocessing must not silently clip or remove it.
10. **Identifier provenance needs a separate correction before publication.** Duplicate PubChem CID mappings were found, but identifiers are not model inputs and are not used by this preprocessing pipeline.

These findings motivate a deliberately conservative preprocessing strategy: preserve the chemistry, avoid global transformations, and fit every learned transformation only on the training partition that is allowed to see it.

## Preprocessing decisions at a glance

| EDA observation | Preprocessing response |
|---|---|
| No missing X10 values | **No imputation** in the main workflow; fail loudly if missing values appear |
| Very different feature scales | `StandardScaler` for scale-sensitive X10 models |
| Long tails / extreme valid molecules | **No row deletion, winsorization, or clipping** by default |
| Sparse and strongly predictive `n_OP` | Keep in X10; pre-specify `X10 - n_OP` as an ablation |
| `NumHDonors` had prior exploratory ablation evidence | Keep in X10; pre-specify `X10 - NumHDonors` as a separate ablation |
| TPSA / polar-SASA redundancy | Keep both; do not remove features solely from pairwise correlation |
| PCA may discard useful lower-variance signal | **No PCA in the main representation** |
| Moderate class imbalance | No SMOTE / undersampling here; use model-side class weighting where appropriate |
| Structural cluster dependence | Reuse the five frozen `STRICT_CV_FOLD` assignments |
| TEST structural novelty | Never fit scalers / PCA / feature transforms on TEST |
| Classical-vs-quantum kernel comparison | Both matched kernels must receive the **same preprocessed molecular coordinates** |

If a preprocessing choice is later selected because it improves cross-validation performance—for example `StandardScaler` vs `RobustScaler`, or a PCA dimensionality—then that choice becomes a **hyperparameter** and must be selected inside the inner model-selection loop.

## 1. Setup and reproducibility

The notebook locates the repository root automatically so that it can be run either from `notebooks/` or from the repository root.

The raw file is expected at:

`data/raw/master.csv`

No package installation is performed inside the notebook; the repository's `requirements.txt` should define the environment.

In [1]:
from pathlib import Path
import hashlib
import json
import random

import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for root in candidates:
        if (root / "data" / "raw" / "master.csv").exists():
            return root
    raise FileNotFoundError(
        "Could not locate data/raw/master.csv from the current working directory."
    )

REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "raw" / "master.csv"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Raw dataset:", DATA_PATH)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Repository root: a:\Uni\PaperQuant\BeeQ
Raw dataset: a:\Uni\PaperQuant\BeeQ\data\raw\master.csv
scikit-learn: 1.9.0
pandas: 2.3.3
numpy: 2.3.4


### Reproducibility check: raw-data hash

The handoff package records the SHA-256 hash of the curated `master.csv`. Checking it here protects the modeling pipeline from silently running on a modified data file.

If the dataset is intentionally corrected later—for example after fixing PubChem CID provenance—the expected hash must be updated deliberately and the change documented.

In [2]:
EXPECTED_MASTER_SHA256 = "a73800d9123caf4e7dfc0a5833b2fb3c7d4514b3593e1b7d9974994ed54eab96"

master_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

print("Observed SHA-256:", master_sha256)
print("Expected SHA-256:", EXPECTED_MASTER_SHA256)

if master_sha256 != EXPECTED_MASTER_SHA256:
    print(
        "\nWARNING: master.csv does not match the frozen handoff hash. "
        "Confirm that this is an intentional dataset revision before modeling."
    )
else:
    print("\nHash matches the frozen handoff dataset.")

Observed SHA-256: a73800d9123caf4e7dfc0a5833b2fb3c7d4514b3593e1b7d9974994ed54eab96
Expected SHA-256: a73800d9123caf4e7dfc0a5833b2fb3c7d4514b3593e1b7d9974994ed54eab96

Hash matches the frozen handoff dataset.


## 2. Load the dataset and define the modeling columns

Only the ten molecular descriptors are predictors.

Identifiers (`ID`, `name`, `CID`, `CAS`, `SMILES`), split metadata (`SET`, `STRICT_CV_FOLD`, `BUTINA_CLUSTER_ID`), and pesticide-use metadata are **not** model features.

`SMILES` may be used later to build a separate molecular fingerprint / molecular-encoder benchmark, but it is not part of X10.

In [3]:
df = pd.read_csv(DATA_PATH)

X10 = [
    "MolLogP",
    "MolWt",
    "TPSA_SP",
    "NumHDonors",
    "NumRotatableBonds",
    "NumAromaticRings",
    "nHalogen",
    "n_OP",
    "LiPHEX_prediction",
    "sasa002_frac_polar_hetero_only",
]

CONTINUOUS_FEATURES = [
    "MolLogP",
    "MolWt",
    "TPSA_SP",
    "LiPHEX_prediction",
    "sasa002_frac_polar_hetero_only",
]

COUNT_FEATURES = [
    "NumHDonors",
    "NumRotatableBonds",
    "NumAromaticRings",
    "nHalogen",
    "n_OP",
]

dev = df[df["SET"] == "DEVELOPMENT"].copy()
test = df[df["SET"] == "TEST"].copy()

print("Full dataset:", df.shape)
print("DEVELOPMENT:", dev.shape)
print("TEST:", test.shape)
print("X10 feature count:", len(X10))

Full dataset: (893, 19)
DEVELOPMENT: (712, 19)
TEST: (181, 19)
X10 feature count: 10


## 3. Hard integrity checks before preprocessing

These are not optional cleaning operations. They are **assertions about the dataset we believe we are modeling**.

The notebook should fail rather than silently repair:

- missing X10 values;
- non-finite X10 values;
- unexpected split names;
- missing DEVELOPMENT fold assignments;
- structural-cluster overlap between DEVELOPMENT and TEST;
- or a Butina cluster spread across multiple DEVELOPMENT folds.

This is safer than adding generic imputers or split logic that could hide an upstream data problem.

In [4]:
assert set(df["SET"].unique()) == {"DEVELOPMENT", "TEST"}

assert dev[X10].isna().sum().sum() == 0, "Missing X10 values in DEVELOPMENT."
assert test[X10].isna().sum().sum() == 0, "Missing X10 values in TEST."

assert np.isfinite(dev[X10].to_numpy(dtype=float)).all()
assert np.isfinite(test[X10].to_numpy(dtype=float)).all()

assert dev["STRICT_CV_FOLD"].notna().all()
assert test["STRICT_CV_FOLD"].isna().all()

fold_ids = sorted(dev["STRICT_CV_FOLD"].astype(int).unique().tolist())
assert fold_ids == [1, 2, 3, 4, 5]

dev_clusters = set(dev["BUTINA_CLUSTER_ID"])
test_clusters = set(test["BUTINA_CLUSTER_ID"])
assert dev_clusters.isdisjoint(test_clusters)

cluster_to_fold_count = (
    dev.groupby("BUTINA_CLUSTER_ID")["STRICT_CV_FOLD"]
    .nunique()
)
assert cluster_to_fold_count.max() == 1

assert df["ID"].is_unique
assert df["SMILES"].is_unique
assert df["CAS"].is_unique

print("All preprocessing integrity checks passed.")
print("Frozen folds:", fold_ids)
print("DEVELOPMENT / TEST Butina-cluster overlap:", len(dev_clusters & test_clusters))

if not df["CID"].is_unique:
    print(
        "\nProvenance warning: CID is not unique. "
        "This is known from Phase 0 and must be resolved before publication. "
        "CID is not used as a model feature."
    )

All preprocessing integrity checks passed.
Frozen folds: [1, 2, 3, 4, 5]
DEVELOPMENT / TEST Butina-cluster overlap: 0

Provenance warning: CID is not unique. This is known from Phase 0 and must be resolved before publication. CID is not used as a model feature.


### Fold inventory

The folds are **not regenerated**. Their structural independence is part of the experimental design, even though their exact class and `n_OP` proportions are not identical.

This table is for auditability and later interpretation of fold-to-fold model variance.

In [5]:
fold_inventory = (
    dev.assign(STRICT_CV_FOLD=dev["STRICT_CV_FOLD"].astype(int))
    .groupby("STRICT_CV_FOLD")
    .agg(
        N=("LABEL", "size"),
        toxic_rate=("LABEL", "mean"),
        n_OP_positive_rate=("n_OP", lambda s: (s > 0).mean()),
        structural_clusters=("BUTINA_CLUSTER_ID", "nunique"),
    )
)

fold_inventory

,N,toxic_rate,n_OP_positive_rate,structural_clusters
STRICT_CV_FOLD,,,,
1,137,0.291971,0.065693,75
2,105,0.352381,0.123810,70
3,147,0.251701,0.102041,79
4,181,0.325967,0.099448,74
5,142,0.345070,0.232394,76


## 4. Pre-specified feature sets

The **main representation remains X10**.

EDA is not used to repeatedly search for a smaller feature set. Instead, only two scientifically motivated ablations are declared before the main model comparison:

- **X9-HDonor:** remove `NumHDonors`, because earlier DEVELOPMENT-only ablation work suggested it might modestly hurt several classical models.
- **X9-noOP:** remove `n_OP`, because Phase 0 showed that the organophosphorus motif is unusually predictive and can dominate part of the geometry.

These are **ablation experiments**, not replacements silently chosen after seeing results.

In [6]:
FEATURE_SETS = {
    "X10": X10,
    "X9_HDONOR_ABLATION": [f for f in X10 if f != "NumHDonors"],
    "X9_NO_OP_ABLATION": [f for f in X10 if f != "n_OP"],
}

pd.DataFrame(
    {
        "feature_set": list(FEATURE_SETS.keys()),
        "n_features": [len(v) for v in FEATURE_SETS.values()],
        "features": [", ".join(v) for v in FEATURE_SETS.values()],
    }
)

,feature_set,n_features,features
0,X10,10,"MolLogP, MolWt, TPSA_SP, NumHDonors, NumRotata..."
1,X9_HDONOR_ABLATION,9,"MolLogP, MolWt, TPSA_SP, NumRotatableBonds, Nu..."
2,X9_NO_OP_ABLATION,9,"MolLogP, MolWt, TPSA_SP, NumHDonors, NumRotata..."


## 5. Missing-value policy: no imputation in the current dataset

Because all X10 values are complete, the main pipeline does **not** contain an imputer.

Why not add one “just in case”?

1. It is unnecessary for the current frozen dataset.
2. An unexpected missing value should trigger an upstream data-quality investigation rather than be silently replaced.
3. If a future external dataset contains missing descriptors, the imputation method would itself become part of the modeling protocol and must be fitted using training data only.

For the current study, the preprocessing contract is therefore:

> **Missing X10 value -> stop and investigate.**

## 6. Outliers and skewness: preserve valid chemistry

Phase 0 showed long-tailed descriptors and a particularly extreme TEST molecule, Chitosan hydrochloride.

No molecule is removed, clipped, or winsorized simply because it is statistically unusual.

Why:

- chemical datasets legitimately contain molecules with unusual size, polarity, or atom counts;
- removing extremes would change the intended applicability domain;
- clipping based on the full dataset could leak information from validation or TEST;
- and the extreme TEST compound is scientifically useful for later applicability-domain / uncertainty analysis.

The default transformation for scale-sensitive models is therefore **standardization without clipping**.

`RobustScaler` is defined below only as a pre-specified sensitivity option. If it is chosen because it improves model performance, that choice must be made inside nested model selection rather than from TEST.

## 7. Scaling policy by model family

Different model families need different treatment.

| Model family | X10 preprocessing |
|---|---|
| Logistic regression | StandardScaler |
| Linear SVM | StandardScaler |
| RBF-SVM | StandardScaler |
| X10 kNN | StandardScaler |
| MLP | StandardScaler |
| QSVM / quantum kernel on X10 | Same fold-fitted scaled coordinates as its matched classical kernel comparator |
| Random Forest / Extra Trees | Raw X10 (`passthrough`) |
| CatBoost / tree boosting | Raw X10 by default |
| Morgan/ECFP + Tanimoto | Separate structural representation; X10 scaling does not apply |

Standardization does **not** mean that integer count descriptors become continuous measurements conceptually. It only puts their numerical contributions onto a comparable scale for algorithms based on distances, margins, or gradient optimization.

The main kernel comparison should not give the quantum model a hidden preprocessing advantage. If a later quantum feature map requires an additional deterministic angle mapping, the matched classical kernel experiment must receive the same coordinates before kernel construction, or the difference must be explicitly labeled as model-specific encoding rather than a pure kernel comparison.

In [7]:
def make_preprocessor(policy="standard"):
    # Return an unfitted preprocessing object.
    #
    # policy:
    #   - 'standard': StandardScaler; main choice for scale-sensitive X10 models
    #   - 'robust': RobustScaler; sensitivity candidate only
    #   - 'none': passthrough; appropriate for tree-based models
    if policy == "standard":
        return StandardScaler()
    if policy == "robust":
        return RobustScaler()
    if policy == "none":
        return "passthrough"
    raise ValueError(f"Unknown preprocessing policy: {policy}")

for policy in ["standard", "robust", "none"]:
    print(policy, "->", make_preprocessor(policy))

standard -> StandardScaler()
robust -> RobustScaler()
none -> passthrough


## 8. The central anti-leakage rule: fit preprocessing inside each fold

This is the most important implementation rule in the notebook.

For outer fold \(k\):

1. select the other four folds as the training portion;
2. **fit** the scaler using only those training molecules;
3. transform the training portion;
4. transform fold \(k\) using the already-fitted scaler;
5. never recompute means, standard deviations, PCA directions, or other learned transformations using the validation fold.

The same principle applies again inside any **inner CV** used for hyperparameter optimization.

A globally standardized DEVELOPMENT matrix must **not** be created once and reused across CV folds, because each validation fold would then influence its own representation.

In [8]:
def get_outer_fold(data, fold, feature_set="X10"):
    # Return one frozen structure-aware outer split.
    # No random splitting is performed.
    fold = int(fold)
    features = FEATURE_SETS[feature_set]

    train_part = data[data["STRICT_CV_FOLD"].astype(int) != fold].copy()
    valid_part = data[data["STRICT_CV_FOLD"].astype(int) == fold].copy()

    train_clusters = set(train_part["BUTINA_CLUSTER_ID"])
    valid_clusters = set(valid_part["BUTINA_CLUSTER_ID"])
    assert train_clusters.isdisjoint(valid_clusters)

    X_train = train_part[features].copy()
    y_train = train_part["LABEL"].copy()
    X_valid = valid_part[features].copy()
    y_valid = valid_part["LABEL"].copy()

    return X_train, y_train, X_valid, y_valid, train_part, valid_part

### Demonstration on Fold 1

The following cell deliberately shows that:

- transformed **training** columns have mean ≈ 0 and standard deviation ≈ 1;
- transformed **validation** columns generally do not.

That is the expected signature of leakage-safe standardization. If the validation columns were also forced to mean zero, the validation data would have influenced the transformation.

In [9]:
X_train, y_train, X_valid, y_valid, train_rows, valid_rows = get_outer_fold(
    dev,
    fold=1,
    feature_set="X10",
)

scaler_fold1 = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler_fold1.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns,
)

X_valid_scaled = pd.DataFrame(
    scaler_fold1.transform(X_valid),
    index=X_valid.index,
    columns=X_valid.columns,
)

fold1_transform_audit = pd.DataFrame({
    "train_mean_after_scaling": X_train_scaled.mean(),
    "train_std_after_scaling": X_train_scaled.std(ddof=0),
    "validation_mean_after_scaling": X_valid_scaled.mean(),
    "validation_std_after_scaling": X_valid_scaled.std(ddof=0),
    "training_mean_used_by_scaler": scaler_fold1.mean_,
    "training_scale_used_by_scaler": scaler_fold1.scale_,
})

print("Fold 1 training molecules:", len(X_train))
print("Fold 1 validation molecules:", len(X_valid))
display(fold1_transform_audit)

Fold 1 training molecules: 575
Fold 1 validation molecules: 137


,train_mean_after_scaling,train_std_after_scaling,validation_mean_after_scaling,validation_std_after_scaling,training_mean_used_by_scaler,training_scale_used_by_scaler
MolLogP,1.544658e-17,1.0,-0.004789,1.091564,2.964293,1.839788
MolWt,-1.730017e-16,1.0,-0.013623,1.225940,311.354896,106.487123
TPSA_SP,2.965744e-16,1.0,-0.153978,0.905584,75.162870,46.848376
NumHDonors,2.471453e-17,1.0,0.097113,1.426556,0.820870,1.168090
NumRotatableBonds,3.089316e-17,1.0,-0.048565,1.012308,4.566957,2.956841
NumAromaticRings,1.235726e-16,1.0,-0.057036,0.959709,1.286957,0.935904
nHalogen,1.235726e-17,1.0,-0.148649,0.837218,1.302609,1.986628
n_OP,0.000000e+00,1.0,-0.212173,0.668303,0.144348,0.370708
LiPHEX_prediction,1.853590e-17,1.0,-0.094706,0.856074,0.328131,1.402621
sasa002_frac_polar_hetero_only,2.595026e-16,1.0,-0.248220,0.819863,0.170723,0.091095


#### Interpretation

The training portion is centered and scaled because those molecules were used to fit `StandardScaler`.

The validation portion is **not** centered to zero and does not have unit variance—and that is correct. It is being observed through a transformation learned without access to its own distribution.

This same pattern must hold for every outer fold and for every inner fold used later during hyperparameter selection.

## 9. Audit fold-local scaling across all five frozen folds

This section checks the preprocessing mechanics for every outer fold before any classifier is introduced.

For each fold we verify:

- zero structural-cluster overlap;
- finite transformed matrices;
- training means ≈ 0;
- training standard deviations ≈ 1.

This makes preprocessing a separately testable component of the experiment.

In [10]:
fold_preprocessing_audit = []

for fold in [1, 2, 3, 4, 5]:
    X_train, y_train, X_valid, y_valid, train_rows, valid_rows = get_outer_fold(
        dev,
        fold=fold,
        feature_set="X10",
    )

    scaler = StandardScaler()
    X_train_t = scaler.fit_transform(X_train)
    X_valid_t = scaler.transform(X_valid)

    assert np.isfinite(X_train_t).all()
    assert np.isfinite(X_valid_t).all()

    train_clusters = set(train_rows["BUTINA_CLUSTER_ID"])
    valid_clusters = set(valid_rows["BUTINA_CLUSTER_ID"])

    fold_preprocessing_audit.append({
        "fold": fold,
        "train_n": len(X_train),
        "valid_n": len(X_valid),
        "train_positive_rate": y_train.mean(),
        "valid_positive_rate": y_valid.mean(),
        "cluster_overlap": len(train_clusters & valid_clusters),
        "max_abs_train_mean": np.abs(X_train_t.mean(axis=0)).max(),
        "max_abs_train_std_minus_1": np.abs(X_train_t.std(axis=0) - 1).max(),
        "max_abs_validation_value": np.abs(X_valid_t).max(),
    })

fold_preprocessing_audit = pd.DataFrame(fold_preprocessing_audit)
fold_preprocessing_audit

,fold,train_n,valid_n,train_positive_rate,valid_positive_rate,cluster_overlap,max_abs_train_mean,max_abs_train_std_minus_1,max_abs_validation_value
0,1,575,137,0.316522,0.291971,0,2.965744e-16,1.110223e-16,9.570437
1,2,607,105,0.304778,0.352381,0,9.949939e-17,2.220446e-16,8.731876
2,3,565,147,0.327434,0.251701,0,2.263676e-16,1.110223e-16,5.816012
3,4,531,181,0.306968,0.325967,0,3.027501e-16,2.220446e-16,5.168174
4,5,570,142,0.303509,0.345070,0,1.869849e-16,2.220446e-16,8.866914


## 10. PCA policy: not part of the main preprocessing pipeline

Phase 0 showed that PCA is useful for **understanding** X10 but not automatically for reducing it.

The main modeling representation therefore remains the original ten descriptors.

PCA should only be introduced if there is a specific experimental reason—for example, a quantum resource constraint. In that case:

- scaling and PCA must both be fitted inside the training fold;
- the number of components must be pre-specified or tuned inside inner CV;
- the same PCA coordinates must be supplied to the matched classical and quantum kernel comparison.

The helper below shows the correct order without selecting any dimensionality here.

In [11]:
def make_scaled_pca_pipeline(n_components):
    return Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(
            n_components=n_components,
            svd_solver="full",
        )),
    ])

example_pca_pipeline = make_scaled_pca_pipeline(n_components=5)
example_pca_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scale', ...), ('pca', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",5
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPAC

## 11. Class imbalance is **not** solved by data synthesis in preprocessing

DEVELOPMENT contains about 31% toxic molecules. This is moderate imbalance, but the preprocessing notebook does not:

- generate SMOTE samples;
- duplicate minority molecules;
- discard majority molecules;
- or alter the target distribution.

Synthetic interpolation in X10 does not necessarily correspond to a chemically realizable molecule and would complicate the interpretation of the structure-aware evaluation.

Instead, imbalance is handled later through:

- model-side class weighting where supported;
- metrics that are meaningful under imbalance (`AUROC`, `AUPRC`, `MCC`, balanced accuracy);
- and threshold selection performed only from training / inner-CV predictions.

`class_weight="balanced"` is therefore a **model configuration**, not a preprocessing transformation.

## 12. Final DEVELOPMENT → historical TEST transformation

The historical TEST set must never be used to **fit** a preprocessing transformation.

After the model family, hyperparameters, feature set, and preprocessing policy have been locked using DEVELOPMENT only, the final workflow is:

1. fit the selected preprocessor on **all DEVELOPMENT** molecules;
2. transform DEVELOPMENT;
3. transform TEST using that fitted object;
4. fit the final model on transformed DEVELOPMENT;
5. evaluate once on the historical TEST benchmark.

The code below demonstrates only the transformation mechanics. It does **not** access TEST labels and does not evaluate a model.

In [12]:
final_standard_scaler = StandardScaler()

X_dev_final = dev[X10].copy()
X_test_final = test[X10].copy()

X_dev_final_scaled = final_standard_scaler.fit_transform(X_dev_final)
X_test_final_scaled = final_standard_scaler.transform(X_test_final)

assert np.isfinite(X_dev_final_scaled).all()
assert np.isfinite(X_test_final_scaled).all()

print("Final DEVELOPMENT transformed shape:", X_dev_final_scaled.shape)
print("Historical TEST transformed shape:", X_test_final_scaled.shape)
print("TEST labels were not accessed in this transformation.")

Final DEVELOPMENT transformed shape: (712, 10)
Historical TEST transformed shape: (181, 10)
TEST labels were not accessed in this transformation.


### Why we do **not** save one globally scaled CSV for cross-validation

A single scaled DEVELOPMENT CSV would encourage a subtle leakage error:

> scale all 712 DEVELOPMENT molecules first -> then split into folds.

That lets every validation fold influence the feature means and variances used to represent itself.

Therefore this repository should keep the canonical molecular descriptors unscaled and let each model pipeline fit its own fold-local preprocessor.

A static transformed matrix is only meaningful for a **specific fitted preprocessing object**, such as the final all-DEVELOPMENT scaler used after model selection is complete.

## 13. Preprocessing manifest for the repository

Instead of exporting a globally scaled matrix, this notebook writes a small machine-readable manifest to:

`data/processed/preprocessing_manifest.json`

The manifest records the dataset hash, feature sets, seed, frozen folds, and preprocessing policy. It is an audit artifac-not a transformed dataset.

In [13]:
preprocessing_manifest = {
    "schema_version": 1,
    "dataset": {
        "path": "data/raw/master.csv",
        "sha256": master_sha256,
        "n_total": int(len(df)),
        "n_development": int(len(dev)),
        "n_test": int(len(test)),
    },
    "seed": SEED,
    "main_feature_set": "X10",
    "feature_sets": FEATURE_SETS,
    "frozen_cv_folds": [1, 2, 3, 4, 5],
    "policies": {
        "missing_values": "No imputation; fail if X10 is missing or non-finite.",
        "outliers": "No automatic deletion, clipping, or winsorization.",
        "scale_sensitive_models": "StandardScaler fitted within the allowed training fold only.",
        "tree_models": "Raw X10 / passthrough by default.",
        "robust_scaler": "Sensitivity candidate only; if selected by performance, tune inside inner CV.",
        "pca": "Not part of main X10 pipeline; optional fold-local ablation/resource experiment.",
        "class_balance": "No over/undersampling in preprocessing; handle with model-side weighting and metrics.",
        "cv": "Reuse frozen STRICT_CV_FOLD assignments; never regenerate for favorable performance.",
        "test": "Never fit transformations on TEST; historical TEST labels are not used during preprocessing.",
        "kernel_fairness": "Matched classical and quantum kernels receive the same preprocessed coordinates.",
    },
}

manifest_path = PROCESSED_DIR / "preprocessing_manifest.json"
manifest_path.write_text(
    json.dumps(preprocessing_manifest, indent=2),
    encoding="utf-8",
)

print("Wrote:", manifest_path.relative_to(REPO_ROOT))

Wrote: data\processed\preprocessing_manifest.json


## 14. Preprocessing contract for the modeling notebooks

The next model notebooks should treat the following rules as fixed unless a change is explicitly documented and evaluated inside the correct CV layer:

1. **Main input:** X10.
2. **Frozen evaluation units:** the five existing `STRICT_CV_FOLD`s.
3. **No missing-value imputation** for the current frozen data.
4. **No automatic molecule removal or clipping** based on descriptor extremes.
5. **Scale-sensitive models:** `StandardScaler`, fitted separately within each training fold.
6. **Tree models:** raw X10 by default.
7. **No globally pre-scaled CV matrix.**
8. **No PCA in the main analysis.** PCA is a separate, fold-local experiment if needed.
9. **No SMOTE, random oversampling, or undersampling** in preprocessing.
10. **Pre-specified feature ablations only:** `X9_HDONOR_ABLATION` and `X9_NO_OP_ABLATION`.
11. **No TEST-driven preprocessing decisions.**
12. **Any preprocessing option chosen based on predictive performance is a hyperparameter** and belongs inside inner CV.
13. **Matched RBF/QSVM experiments must use the same molecular coordinates.**
14. **Known provenance issues remain visible:** duplicated CID mappings must be corrected before publication; identifiers are never model features.

### Phase 1 stopping point

At this point preprocessing is deliberately simple. That is a strength, not a missing step: the EDA did not justify aggressive transformations.

The next phase should implement **nested, structure-aware model selection** around these preprocessing rules rather than continuing to alter the representation after observing model or TEST performance.